# Prompt effect: our prompt vs. WebOrganizer's prompt

Reproduces **section 3, "Prompt effect"**, of
[`docs/label_distribution_pipeline.md`](../docs/label_distribution_pipeline.md).

**Question:** does our prompt change the labels compared with WebOrganizer's prompt?

**Design:** the same model (Llama-3.1-8B-Instruct) labelled the same documents twice:

- **WebOrganizer's prompt:** WebOrganizer's own annotations,
  [`WebOrganizer/TopicAnnotations-Llama-3.1-8B`](https://huggingface.co/datasets/WebOrganizer/TopicAnnotations-Llama-3.1-8B).
- **Our prompt:** `generate_label_distributions.py` run on the same documents in the
  same order, with WebOrganizer's 50,000-character truncation
  (`results/distribution/weborganizer-comparison/Llama-3.1-8B-Instruct/topic.jsonl`).

With model and documents fixed, the remaining differences are the prompt and minor
implementation details: the inference engine (SGLang vs vLLM) and each document's
label order. Section 7 compares topic shares on Nemotron-CC, to separate the prompt
effect from the effect of the data.

**Status (2 October 2026):** sections 1–5, and the WebOrganizer rows of section 6, use
the run on WebOrganizer's documents, which is unchanged, so they still reproduce the
report. The Nemotron-CC rows of sections 6 and 7 now use the current runs (2,000
documents, 26 labels, rotated prompt order) for all five candidate models: the report's
10,000-document runs have been replaced, so those numbers differ from the report.
Missing runs are skipped with a message.

**Sections:** 1. load and join · 2. confidence · 3. agreement · 4. topic shares ·
5. disagreements · 6. letter bias · 7. data effect · 8. summary against the report.

**Running it:** on Roihu, `module load python-vllm/0.19.1` provides pandas,
matplotlib and JupyterLab. The first run downloads WebOrganizer's label fields (no
document text) through the Hugging Face dataset viewer API, which needs internet
access, and caches them in `data/weborganizer_samples/`; later runs read the cache.

**Common modifications** (all in the parameter cell):

- *Use all documents instead of the report's 3,072:* set `N_DOCS = None`.
- *Compare another model or run:* point `OURS_PATH` at its output and `WO_DATASET` at
  the matching WebOrganizer dataset. The run must have streamed that dataset from its
  first row.
- *Format labels instead of topics:* use a format run, a `FormatAnnotations-...`
  dataset and `formats.yaml` as `LABELS_PATH`, and update the label names in the
  section 7 plot.
- *A run with the added labels (`Other`, `Bad example`):* nothing to change; they are
  routed back to WebOrganizer's 24 labels for every comparison with WebOrganizer.

## Setup

This cell imports everything the notebook needs and defines two small helpers.

- **Repository root:** the notebook works whether Jupyter was started in the
  repository root or in `notebooks/`, by locating the root from the current directory.
- **Shared code instead of reimplementing metrics:** loading, validation and metrics come
  from `scripts/data_analysis/distribution_metrics.py`, and WebOrganizer rows are
  converted with `reference_record` from `evaluate_distribution_reference.py`. The
  notebook therefore computes exactly what the evaluation scripts compute; if a metric
  changes there, it changes here too.
- **`weborganizer_samples.fetch_rows`** downloads and caches WebOrganizer's label fields.
- **Added labels:** our taxonomies add `Other` and `Bad example`, which WebOrganizer's
  data doesn't have. `weborganizer_label_subset` splits our label file into
  WebOrganizer's 24 labels and the added ones, and `restrict_to_labels` routes the
  added labels back for comparisons: their probability is removed and the rest is
  renormalized, as `evaluate_distribution_reference.py` does. The report's runs were
  made before the labels were added and have 24 labels; runs made since have 26.
- **Helpers:** `quantile` defines percentiles as the element at index `int(q * n)` of the
  sorted values, the definition used for the report; `top_prob` is the probability of a
  document's most likely label.

In [ ]:
from __future__ import annotations

import math
import statistics
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Find the repository root whether Jupyter runs from the root or from notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts" / "data_analysis"))

# The same loading and metrics as the evaluation scripts, so the numbers agree.
from distribution_metrics import (
    cohen_kappa,           # chance-corrected agreement between two sets of top labels
    entropy_bits,          # entropy of one distribution, in bits
    jensen_shannon_bits,   # divergence between two distributions, 0 (same) to 1 bit
    letter_bias,           # tests whether the shown letter predicts the chosen label
    probabilities,         # log-probabilities -> probabilities
    ranked_indices,        # label indices from most to least probable
    read_distribution_file,
    restrict_to_labels,    # drop labels and renormalize (routes added labels back)
    top_confusions,        # most frequent (reference, candidate) disagreements
    top_index,             # index of the most probable label
)
from evaluate_distribution_reference import load_label_names, reference_record, weborganizer_label_subset
from weborganizer_samples import fetch_rows

# Answer letters in the order they were shown; position 0 = "A". Runs with 24
# labels use A-X, runs with the added labels use A-Z.
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def quantile(values, q):
    """Return the q-quantile as the element at index int(q * n) of the sorted values."""
    ordered = sorted(values)
    return ordered[min(len(ordered) - 1, int(q * len(ordered)))]


def top_prob(record):
    """Return the probability of the record's most likely label."""
    return math.exp(max(record.logprobs))

## Parameters

Everything a typical rerun changes is set here.

- `N_DOCS`: how many documents to compare, taken from the start of our run. The
  default, 3,072, is the number our run had finished when the report was written, so
  the defaults reproduce the report exactly. `None` uses all documents in the file.
- `OURS_PATH`: our run on WebOrganizer's documents.
- `WO_DATASET`: the WebOrganizer annotations for the same documents.
- `LABELS_PATH`: our label file: WebOrganizer's labels in taxonomy order, followed by
  the added labels. The loading code checks the order against every WebOrganizer row.
- `CACHE_DIR`: where downloaded WebOrganizer rows are cached.
- `NEMOTRON_RUNS`: our runs on English Nemotron-CC, used in sections 6 and 7. Files
  that don't exist are skipped with a message.

In [ ]:
# 3072 reproduces the report: the comparison run had 3,072 documents when it was
# written. None uses every document in the file (10,000).
N_DOCS = 3072

OURS_PATH = ROOT / "results/distribution/weborganizer-comparison/Llama-3.1-8B-Instruct/topic.jsonl"
WO_DATASET = "WebOrganizer/TopicAnnotations-Llama-3.1-8B"
LABELS_PATH = ROOT / "scripts/data_processing/topics.yaml"
CACHE_DIR = ROOT / "data/weborganizer_samples"
NEMOTRON_RUNS = {
    model: ROOT / f"results/distribution/{name}-annotations/topic/english-parallel.jsonl"
    for model, name in {
        "Llama-3.1-8B": "Llama-3.1-8B-Instruct",
        "Qwen3.8-27B": "Qwen3.8-27B",
        "Gemma-4-31B": "gemma-4-31B-it",
        "Qwen3.5-122B-A10B": "Qwen3.5-122B-A10B",
        "Mistral-Medium-3.5-128B": "Mistral-Medium-3.5-128B",
    }.items()
}

## 1. Load both runs and join them by document

### Our run

`read_distribution_file` validates every record (complete label set, finite
log-probabilities that sum to one, no duplicate IDs) and returns each document's
log-probabilities in label-file order, plus the letter each label was shown under.
We keep the first `N_DOCS` documents in file order.

The run may have WebOrganizer's 24 labels or all 26 of ours; anything else stops
the notebook.

In [ ]:
all_names, all_labels = load_label_names(LABELS_PATH)
# label_names / labels: WebOrganizer's 24 labels (full and short names);
# added_labels: the labels WebOrganizer doesn't have.
label_names, labels, added_labels = weborganizer_label_subset(all_names, all_labels)
ours_labels, ours_all = read_distribution_file(OURS_PATH)
assert ours_labels in (labels, all_labels), "our run's labels match neither label set"
doc_ids = list(ours_all)[:N_DOCS]
print(f"Our run: {len(ours_all)} documents in the file; using the first {len(doc_ids)}")

### WebOrganizer's labels for the same documents

Our run streamed WebOrganizer's dataset from its first row, so our documents are the
dataset's first rows. We fetch those rows in pages of 100 (the API's maximum) until
all of our documents are covered.

`reference_record` turns each row into the same format as our records:

- `choice_loss` (WebOrganizer's log-probability per label) is normalised into
  log-probabilities that sum to one.
- `label_permutation` gives the letter each label was shown under.
- The row's `top_choice` text is checked against our label names, so a different
  label order fails loudly instead of silently swapping labels. WebOrganizer's
  taxonomy misspells "Hardware" as "Hardare"; that one difference is allowed.

In [ ]:
offsets = list(range(0, math.ceil(len(doc_ids) / 100) * 100, 100))
rows = fetch_rows(
    WO_DATASET,
    offsets,
    CACHE_DIR / f"TopicAnnotations-Llama-3.1-8B_first{len(offsets) * 100}.jsonl",
)
# Key WebOrganizer's records by WARC record ID, the ID our run used for these documents.
wo_all = {}
for number, row in enumerate(rows, start=1):
    record = reference_record(row, label_names, number)
    wo_all[record.doc_id] = record
print(f"WebOrganizer: {len(wo_all)} rows loaded")

### Join

Documents are matched by WARC record ID, which both sides store. Every one of our
documents must be found; otherwise the cell stops, rather than quietly comparing
fewer documents. `ours[k]` and `wo[k]` are then the same document.

If our run has the added labels, `ours` routes them back to WebOrganizer's 24
labels, so both sides compare the same label set. `ours_full` keeps the original
distributions, for the letter-bias analysis in section 6, and the cell reports how
often an added label was our top label.

In [ ]:
missing = [doc_id for doc_id in doc_ids if doc_id not in wo_all]
assert not missing, f"{len(missing)} of our documents are not in the WebOrganizer rows"
ours_full = [ours_all[doc_id] for doc_id in doc_ids]
wo = [wo_all[doc_id] for doc_id in doc_ids]
n = len(ours_full)
if ours_labels == labels:
    ours = ours_full
else:
    keep = [ours_labels.index(label) for label in labels]
    ours = [restrict_to_labels(r, keep) for r in ours_full]
    added = [ours_labels.index(label) for label in added_labels]
    routed_top = sum(top_index(r.logprobs) in added for r in ours_full) / n
    print(f"Routed {', '.join(added_labels)} back to WebOrganizer's labels; they were our top label for {routed_top:.1%} of documents")
print(f"{n} documents joined by WARC record ID")

## 2. Distribution shape

How confident does each prompt make the model? Per prompt:

- **median top prob** and **25th pct top prob:** the probability of the top label;
  the 25th percentile shows the less confident quarter of documents.
- **share top prob ≥ 0.99:** documents where the model is essentially certain.
- **mean entropy:** uncertainty over all 24 labels; 0 bits = certain, 4.58 bits =
  uniform.

Because both prompts labelled the same documents, we also compute the **paired**
entropy difference per document, which removes differences between documents.

In [ ]:
def shape(name, records):
    """Return the confidence summary of one prompt's records."""
    tops = [top_prob(r) for r in records]
    return {
        "prompt": name,
        "median top prob": statistics.median(tops),
        "25th pct top prob": quantile(tops, 0.25),
        "share top prob >= 0.99": sum(t >= 0.99 for t in tops) / len(tops),
        "mean entropy (bits)": statistics.fmean(entropy_bits(r.logprobs) for r in records),
    }


display(pd.DataFrame([shape("WebOrganizer", wo), shape("ours", ours)]).round(3))

# Positive = our prompt is less confident on that document.
entropy_diff = [entropy_bits(o.logprobs) - entropy_bits(w.logprobs) for o, w in zip(ours, wo)]
print(
    f"Paired entropy difference (ours - WebOrganizer): mean {statistics.fmean(entropy_diff):+.3f} bits; "
    f"ours is less confident on {sum(d > 0 for d in entropy_diff) / n:.0%} of documents"
)

**Plot.** *Left:* the distribution of top-label probabilities for each prompt.
Overlapping histograms mean the prompts make the model similarly confident. *Right:*
the per-document entropy difference. A narrow peak at 0 means most documents get
almost the same distribution; spread to the right means our prompt is less confident
on those documents.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
bins = [i / 20 for i in range(21)]
axes[0].hist([top_prob(r) for r in wo], bins=bins, alpha=0.6, label="WebOrganizer prompt")
axes[0].hist([top_prob(r) for r in ours], bins=bins, alpha=0.6, label="our prompt")
axes[0].set_xlabel("top-label probability")
axes[0].set_ylabel("documents")
axes[0].set_title("Confidence in the top label")
axes[0].legend()
axes[1].hist(entropy_diff, bins=40, color="tab:gray")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("entropy difference, ours - WebOrganizer (bits)")
axes[1].set_ylabel("documents")
axes[1].set_title("Paired entropy difference")
fig.tight_layout()
plt.show()

## 3. Agreement

Each document gives a pair (WebOrganizer's top label, our top label).

- **top-1 agreement:** share of documents with the same top label.
- **Cohen's kappa:** agreement corrected for what label frequencies alone would give
  by chance; 1 = perfect, 0 = chance level.
- **WO label in our top 2 / top 3:** whether WebOrganizer's label is at least among
  our most likely labels. This separates near misses from real disagreements.
- **our prob. of WO's label / WO prob. of our label:** the probability each prompt
  gives the other's top label, averaged over documents. Similar values mean neither
  prompt is systematically more one-sided.

In [ ]:
pairs = [(top_index(w.logprobs), top_index(o.logprobs)) for o, w in zip(ours, wo)]
agree = [a == b for a, b in pairs]
ours_ranked = [ranked_indices(o.logprobs) for o in ours]
agreement = {
    "top-1 agreement": sum(agree) / n,
    "Cohen's kappa": cohen_kappa(pairs, len(labels)),
    "WO label in our top 2": sum(a in r[:2] for (a, _), r in zip(pairs, ours_ranked)) / n,
    "WO label in our top 3": sum(a in r[:3] for (a, _), r in zip(pairs, ours_ranked)) / n,
    "our prob. of WO's label (mean)": statistics.fmean(probabilities(o.logprobs)[a] for o, (a, _) in zip(ours, pairs)),
    "WO prob. of our label (mean)": statistics.fmean(probabilities(w.logprobs)[b] for w, (_, b) in zip(wo, pairs)),
}
display(pd.DataFrame([agreement]).round(3))

### Agreement by confidence

If the prompts disagree mainly on genuinely ambiguous documents, agreement should be
low where the model is unsure and high where it is confident. Documents are grouped by
top-label probability, once using WebOrganizer's confidence and once using ours.
The bin edges (0.5, 0.8, 0.99) separate unsure, fairly sure, sure and certain.

In [ ]:
bin_edges = [(0.0, 0.5), (0.5, 0.8), (0.8, 0.99), (0.99, 1.01)]  # 1.01 so p = 1.0 is included
by_confidence = []
for low, high in bin_edges:
    when_wo = [a for a, w in zip(agree, wo) if low <= top_prob(w) < high]
    when_ours = [a for a, o in zip(agree, ours) if low <= top_prob(o) < high]
    by_confidence.append(
        {
            "confidence": f"{low:.2f}-{min(high, 1):.2f}",
            "agreement when WO this confident": sum(when_wo) / len(when_wo) if when_wo else float("nan"),
            "WO docs": len(when_wo),
            "agreement when ours this confident": sum(when_ours) / len(when_ours) if when_ours else float("nan"),
            "our docs": len(when_ours),
        }
    )
display(pd.DataFrame(by_confidence).round(3))

**Plot.** Agreement per confidence bin. Bars that rise from left to right show that
disagreements are concentrated in documents where the model itself is unsure.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
positions = range(len(by_confidence))
ax.bar([p - 0.2 for p in positions], [row["agreement when WO this confident"] for row in by_confidence], width=0.4, label="binned by WebOrganizer's confidence")
ax.bar([p + 0.2 for p in positions], [row["agreement when ours this confident"] for row in by_confidence], width=0.4, label="binned by our confidence")
ax.set_xticks(list(positions))
ax.set_xticklabels([row["confidence"] for row in by_confidence])
ax.set_xlabel("top-label probability")
ax.set_ylabel("top-1 agreement")
ax.set_ylim(0, 1)
ax.set_title("Agreement rises with confidence")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Topic shares

The share of documents whose top label is each topic, under each prompt, sorted by
WebOrganizer's share. Even with per-document disagreements, a prompt that shifts the
overall topic mix would show up here as large differences.

In [ ]:
wo_counts = Counter(a for a, _ in pairs)
ours_counts = Counter(b for _, b in pairs)
order = sorted(range(len(labels)), key=lambda i: -wo_counts[i])  # used by later plots too
share_rows = [
    {
        "label": labels[i],
        "WebOrganizer": wo_counts[i] / n,
        "ours": ours_counts[i] / n,
        "difference": (ours_counts[i] - wo_counts[i]) / n,
    }
    for i in order
]
max_share_difference = max(abs(row["difference"]) for row in share_rows)
display(pd.DataFrame(share_rows).round(3))
print(f"Largest difference: {max_share_difference * 100:.1f} percentage points")

**Plot.** The same shares as bars. Pairs of bars of similar length mean the two
prompts produce the same topic mix.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
rows_y = list(range(len(order)))
ax.barh([y - 0.2 for y in rows_y], [wo_counts[i] / n for i in order], height=0.4, label="WebOrganizer prompt")
ax.barh([y + 0.2 for y in rows_y], [ours_counts[i] / n for i in order], height=0.4, label="our prompt")
ax.set_yticks(rows_y)
ax.set_yticklabels([labels[i] for i in order])
ax.invert_yaxis()  # largest share at the top
ax.set_xlabel("share of documents")
ax.set_title("Topic shares on the same documents")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Most common disagreements

Which label pairs the prompts most often swap. Swaps between neighbouring categories
(e.g. Software vs Software Dev.) point to unclear category boundaries, not to a
problem with either prompt.

In [ ]:
confusions = [
    {"WebOrganizer": c["reference"], "ours": c["candidate"], "documents": c["count"]}
    for c in top_confusions(pairs, labels, limit=10)
]
display(pd.DataFrame(confusions))

## 6. Letter bias

Labels are shuffled per document, so without bias the top label sits at every letter
equally often (1/24 of documents each). From `letter_bias`:

- **winner_chi2:** chi-square test of that; above 35.2 (the 5% critical value for 24
  letters) the letter of the chosen label is unlikely to be chance.
- **runner_up_chi2:** the same test for the second most likely label.
- **winner_mean_position:** the average letter position of the chosen label (A = 0);
  11.5 without bias, lower when early letters are favoured.
- **winner_mean_position_z:** how many standard errors that average is from 11.5.

The first two rows compare the prompts on WebOrganizer's documents. The Nemotron rows
use every document of each run and show whether a preference for early letters is
specific to Llama-8B or affects all models.

Letter bias is measured on the full distributions (`ours_full`), because a routed
distribution can never choose the letters of the removed labels. With 26 labels the
unbiased mean position is 12.5 and the 5% chi-square critical value is 37.7.

In [ ]:
bias_rows = [
    {"run": "WebOrganizer prompt, WO docs", **letter_bias(wo)},
    {"run": "our prompt, WO docs", **letter_bias(ours_full)},
]
for model, path in NEMOTRON_RUNS.items():
    if not path.exists():
        print(f"{model}: {path.relative_to(ROOT)} not found; skipped")
        continue
    _, records = read_distribution_file(path)
    bias_rows.append({"run": f"our prompt, Nemotron, {model}", **letter_bias(records.values())})
display(pd.DataFrame(bias_rows).round(2))

**Plot.** How often the chosen label sat at each letter, for both prompts on the same
documents. Without bias the bars scatter around the dashed line. Taller bars at early
letters and shorter ones at late letters mean early letters are favoured.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
series = (("WebOrganizer prompt", wo), ("our prompt", ours_full))
num_letters = max(records[0].num_letters for _, records in series)
for offset, (name, records) in zip((-0.2, 0.2), series):
    winners = Counter(r.letter_positions[top_index(r.logprobs)] for r in records)
    ax.bar([j + offset for j in range(num_letters)], [winners[j] for j in range(num_letters)], width=0.4, label=name)
# Expected count per letter without bias; one line per number of letters shown.
for letters_shown in sorted({records[0].num_letters for _, records in series}):
    ax.axhline(n / letters_shown, color="black", linestyle="--", linewidth=1, label=f"no bias ({letters_shown} letters)")
ax.set_xticks(range(num_letters))
ax.set_xticklabels(list(LETTERS[:num_letters]))
ax.set_xlabel("letter of the top label")
ax.set_ylabel("documents")
ax.set_title("Llama-3.1-8B: letter of the chosen label, same documents")
ax.set_ylim(0, ax.get_ylim()[1] * 1.25)  # room for the legend above the bars
ax.legend(ncol=3, loc="upper center")
fig.tight_layout()
plt.show()

## 7. Data effect: Nemotron-CC vs. WebOrganizer's documents

Our earlier Nemotron-CC results had very different topic shares from WebOrganizer's.
Section 4 showed that our prompt reproduces WebOrganizer's shares on WebOrganizer's
own documents. If the Nemotron shares still differ, the difference comes from the
corpus, not from the prompt. The Nemotron columns use the first `N_DOCS` documents of
each run; runs with the added labels are routed back to WebOrganizer's labels first.

In [ ]:
nemotron_shares = {}
for model, path in NEMOTRON_RUNS.items():
    if not path.exists():
        print(f"{model}: {path.relative_to(ROOT)} not found; skipped")
        continue
    run_labels, records = read_distribution_file(path)
    keep = [run_labels.index(label) for label in labels]  # identity for 24-label runs
    tops = Counter(top_index(restrict_to_labels(r, keep).logprobs) for r in list(records.values())[:N_DOCS])
    total = sum(tops.values())
    nemotron_shares[model] = [tops[i] / total for i in range(len(labels))]

# One column per run; label index order, so share_columns[name][i] is label i.
share_columns = {
    "WO prompt, WO docs": [wo_counts[i] / n for i in range(len(labels))],
    "our prompt, WO docs": [ours_counts[i] / n for i in range(len(labels))],
    **{f"Nemotron, {model}": values for model, values in nemotron_shares.items()},
}
display(
    pd.DataFrame(
        [{"label": labels[i], **{name: values[i] for name, values in share_columns.items()}} for i in order]
    ).round(3)
)

**Plot.** The labels with the largest differences between the corpora. The first two
bars of each group (both prompts on WebOrganizer's documents) should match each other,
while the Nemotron bars differ from them.

In [ ]:
highlight = ["Science & Tech.", "Health", "History", "Education & Jobs", "Fashion & Beauty", "Travel", "Sports & Fitness", "Home & Hobbies"]
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.8 / len(share_columns)
for k, (name, values) in enumerate(share_columns.items()):
    heights = [values[labels.index(label)] for label in highlight]
    ax.bar([i - 0.4 + width * (k + 0.5) for i in range(len(highlight))], heights, width=width, label=name)
ax.set_xticks(range(len(highlight)))
ax.set_xticklabels(highlight, rotation=30, ha="right")
ax.set_ylabel("share of documents")
ax.set_title("Topic shares: the prompts agree, the corpora differ")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 8. Summary against the report

The key numbers next to the values stated in the report. With the default
`N_DOCS = 3072` they match, apart from the report's rounding. With more documents
the values move slightly, which shows how stable the findings are.

In [ ]:
ours_bias, wo_bias = letter_bias(ours_full), letter_bias(wo)
summary = [
    ("WO prompt: median top prob", shape("", wo)["median top prob"], 0.955),
    ("our prompt: median top prob", shape("", ours)["median top prob"], 0.947),
    ("WO prompt: mean entropy (bits)", shape("", wo)["mean entropy (bits)"], 0.51),
    ("our prompt: mean entropy (bits)", shape("", ours)["mean entropy (bits)"], 0.57),
    ("top-1 agreement", agreement["top-1 agreement"], 0.766),
    ("Cohen's kappa", agreement["Cohen's kappa"], 0.75),
    ("WO label in our top 2", agreement["WO label in our top 2"], 0.914),
    ("WO label in our top 3", agreement["WO label in our top 3"], 0.955),
    ("largest topic-share difference", max_share_difference, 0.012),
    ("our prompt: mean letter position", ours_bias["winner_mean_position"], 10.66),
    ("our prompt: letter position z", ours_bias["winner_mean_position_z"], -6.7),
    ("WO prompt: letter position z", wo_bias["winner_mean_position_z"], -0.9),
]
display(pd.DataFrame(summary, columns=["finding", "this run", "report"]).round(3))